# Notebook 4 — Ablation Studies
**RSNA Intracranial Hemorrhage Detection**

This notebook runs controlled ablation experiments to justify the design decisions
in the main training pipeline.  Each variant trains for a small number of epochs
on a fixed subset so that all experiments fit in one Kaggle session.

### Experiments
| # | Name | Variable | Options |
|---|------|----------|---------|
| A | Architecture | Backbone | EfficientNet-B0 vs ResNet-50 |
| B | Windowing | Preprocessing | 3-window stack vs single raw channel |
| C | Augmentation | Data augmentation | ON vs OFF |

### Required input
- Preprocessing cache (Notebook 02 output) — `manifest.csv` + `cache/` PNGs
- **No pre-trained weights needed** — all variants start from ImageNet init

In [ ]:
# ── 0. Config ──────────────────────────────────────────────────────────────
import os, gc, random, json as _json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Ablation-specific settings ────────────────────────────────────────────
ABLATION_EPOCHS   = 3         # short: just enough to see meaningful differences
ABLATION_SUBSET   = 0.15      # use 15% of data for all ablations (fast + fair)
BATCH_SIZE        = 16
BASE_LR           = 1e-4
SEED              = 42
IMG_SIZE          = 256
NUM_WORKERS       = 4

CACHE_INPUT_DIR   = '/kaggle/input/nb02eda'
MANIFEST_PATH     = f'{CACHE_INPUT_DIR}/manifest.csv'
NPY_CACHE_DIR     = f'{CACHE_INPUT_DIR}/cache'

# CT windows (for 3-channel preprocessing)
WINDOWS = [(40, 80), (75, 215), (40, 380)]

# ─── Load dataset-specific normalization (with ImageNet fallback) ────────
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = _json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Dataset normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print(f'Using ImageNet defaults: mean={MEAN}, std={STD}')

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(SEED)
print(f'Device: {DEVICE}')

In [ ]:
# ── Dataset classes ───────────────────────────────────────────────────────

class ICHDataset(Dataset):
    """Standard dataset: reads cached 3-channel windowed NPY arrays."""
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root  = npy_root
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)                        # uint8 H×W×3 [0,255]
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return self.transform(img), torch.tensor(float(row['any']), dtype=torch.float32)


class ICHDatasetGray(Dataset):
    """Ablation: reads the 3-ch NPY but collapses to 1 channel (brain window only),
    then replicates to 3 channels for compatibility with ImageNet pretrained backbones.
    This simulates training WITHOUT multi-window preprocessing.
    """
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root  = npy_root
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)                        # float32 H×W×3 [0,1]
            img = np.load(path)                        # uint8 H×W×3 [0,255]
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        ch0 = img[:, :, 0]                             # H×W float32
        ch0 = img[:, :, 0]                             # H×W uint8
        img = (img * 255).astype(np.uint8)             # for ToPILImage
        return self.transform(img), torch.tensor(float(row['any']), dtype=torch.float32)


# Transforms
aug_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

no_aug_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

print('Dataset classes defined.')

In [ ]:
# ── Shared utilities ──────────────────────────────────────────────────────
def build_model(arch: str) -> nn.Module:
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    elif arch == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    else:
        raise ValueError(arch)
    return m.to(DEVICE)


def make_loaders(train_df, val_df, train_ds_cls, train_transform, val_transform):
    pos  = int(train_df['any'].sum())
    neg  = len(train_df) - pos
    w    = np.where(train_df['any'].values == 1, neg/pos, 1.0)
    sampler = WeightedRandomSampler(w.tolist(), len(train_df), replacement=True)
    tr_ds = train_ds_cls(train_df, NPY_CACHE_DIR, train_transform)
    vl_ds = ICHDataset(val_df, NPY_CACHE_DIR, val_transform)
    tr_l  = DataLoader(tr_ds, BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    vl_l  = DataLoader(vl_ds, BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return tr_l, vl_l, neg / pos


@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    logits_, labels_ = [], []
    for imgs, lbls in loader:
        with autocast():
            out = model(imgs.to(DEVICE)).squeeze(1).cpu().float()
        logits_.append(out); labels_.append(lbls)
    logits = torch.cat(logits_).numpy()
    labels = torch.cat(labels_).numpy()
    probs  = torch.sigmoid(torch.tensor(logits)).numpy()
    auc    = roc_auc_score(labels, probs)
    fpr, tpr, thr = roc_curve(labels, probs)
    j   = np.argmax(tpr - fpr)
    t   = thr[j]
    prd = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, prd).ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    return dict(auc=round(float(auc),4),
                sensitivity=round(float(sens),4),
                specificity=round(float(spec),4),
                threshold=round(float(t),4))


def run_experiment(name: str, arch: str, train_ds_cls,
                   train_transform, val_transform,
                   train_df, val_df) -> dict:
    print(f'\n{"="*55}')
    print(f'  Experiment: {name}')
    print(f'  Arch: {arch}  |  Epochs: {ABLATION_EPOCHS}')
    print(f'{"="*55}')
    seed_everything(SEED)
    model = build_model(arch)
    tr_l, vl_l, pw = make_loaders(train_df, val_df, train_ds_cls,
                                   train_transform, val_transform)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw]).to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-5)
    scaler    = GradScaler()

    for epoch in range(ABLATION_EPOCHS):
        model.train()
        ep_loss = 0.0
        for imgs, lbls in tqdm(tr_l, desc=f'  Ep {epoch}', leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = criterion(model(imgs).squeeze(1), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            ep_loss += loss.item()
        ep_loss /= len(tr_l)

    result = eval_loader(model, vl_l)
    result.update({'name': name, 'arch': arch})
    print(f'  Result: AUC={result["auc"]:.4f}  '
          f'Sens={result["sensitivity"]:.4f}  '
          f'Spec={result["specificity"]:.4f}')

    # Free GPU memory
    del model, tr_l, vl_l
    gc.collect()
    torch.cuda.empty_cache()
    return result


print('Utilities defined.')

In [ ]:
# ── Load manifest + create ablation subset ────────────────────────────────
manifest = pd.read_csv(MANIFEST_PATH)
ablation_df = manifest.groupby(['split', 'any'], group_keys=False).apply(
    lambda x: x.sample(frac=ABLATION_SUBSET, random_state=SEED)
).reset_index(drop=True)

train_df = ablation_df[ablation_df['split'] == 'train'].reset_index(drop=True)
val_df   = ablation_df[ablation_df['split'] == 'val'].reset_index(drop=True)
print(f'Ablation subset — Train: {len(train_df):,}  Val: {len(val_df):,}')

In [ ]:
# ── Experiment A: Architecture comparison ────────────────────────────────
results = []

res = run_experiment(
    name='A1 EfficientNet-B0 + 3ch windows + augmentation',
    arch='efficientnet_b0',
    train_ds_cls=ICHDataset,
    train_transform=aug_transform,
    val_transform=no_aug_transform,
    train_df=train_df, val_df=val_df
)
results.append(res)

res = run_experiment(
    name='A2 ResNet-50 + 3ch windows + augmentation',
    arch='resnet50',
    train_ds_cls=ICHDataset,
    train_transform=aug_transform,
    val_transform=no_aug_transform,
    train_df=train_df, val_df=val_df
)
results.append(res)

In [ ]:
# ── Experiment B: Windowing ON vs OFF (EfficientNet-B0) ───────────────────
res = run_experiment(
    name='B1 EffNet-B0 + NO windowing (single raw ch replicated)',
    arch='efficientnet_b0',
    train_ds_cls=ICHDatasetGray,   # single channel — no multi-window
    train_transform=aug_transform,
    val_transform=no_aug_transform,
    train_df=train_df, val_df=val_df
)
results.append(res)
# A1 (with windowing) already collected above

In [ ]:
# ── Experiment C: Augmentation ON vs OFF (EfficientNet-B0) ────────────────
res = run_experiment(
    name='C1 EffNet-B0 + 3ch windows + NO augmentation',
    arch='efficientnet_b0',
    train_ds_cls=ICHDataset,
    train_transform=no_aug_transform,   # no augmentation
    val_transform=no_aug_transform,
    train_df=train_df, val_df=val_df
)
results.append(res)
# A1 (with augmentation) already collected above

In [ ]:
# ── Compile results table ─────────────────────────────────────────────────
results_df = pd.DataFrame(results)[['name', 'arch', 'auc', 'sensitivity', 'specificity', 'threshold']]
results_df.columns = ['Experiment', 'Arch', 'Val AUC', 'Sensitivity', 'Specificity', 'Threshold']

print('\n' + '='*90)
print('ABLATION RESULTS')
print('='*90)
print(results_df.to_string(index=False))
print('='*90)

results_df.to_csv('/kaggle/working/ablation_results.csv', index=False)

In [ ]:
# ── Visualise AUC comparison ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ['Val AUC', 'Sensitivity', 'Specificity']
colors = sns.color_palette('muted', len(results_df))

for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.barh(results_df['Experiment'], results_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_xlim(0.5, 1.0)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

plt.suptitle(f'Ablation Study ({ABLATION_EPOCHS} epochs, {ABLATION_SUBSET*100:.0f}% data subset)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/ablation_comparison.png', bbox_inches='tight')
plt.show()

print('\nInterpretation guide:')
print(' A1 vs A2  → justifies architecture selection')
print(' A1 vs B1  → justifies multi-window preprocessing')
print(' A1 vs C1  → justifies data augmentation')

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────
import json as _json_hc

errors = []

# Check results CSV
if not os.path.exists('/kaggle/working/ablation_results.csv'):
    errors.append('ablation_results.csv is MISSING')
else:
    abl_hc = pd.read_csv('/kaggle/working/ablation_results.csv')
    if len(abl_hc) < 4:
        errors.append(f'Only {len(abl_hc)} ablation results — expected 4')
    if 'Val AUC' in abl_hc.columns and abl_hc['Val AUC'].max() < 0.55:
        errors.append(f'All ablation AUCs < 0.55 — possible issue')

# Check comparison plot
if not os.path.exists('/kaggle/working/ablation_comparison.png'):
    errors.append('ablation_comparison.png is MISSING')

health = {
    'notebook': '04_ablations',
    'status'  : 'PASS' if not errors else 'FAIL',
    'errors'  : errors,
    'n_experiments': len(results),
    'best_experiment': max(results, key=lambda x: x['auc'])['name'] if results else 'N/A',
}

with open('/kaggle/working/health_check_nb04.json', 'w') as f:
    _json_hc.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   {len(results)} experiments completed')
    print(f'   Best: {health["best_experiment"]}')